![Module 5: Ambient / Autonomous Mode](../images/module-5-ambient.svg)

# Module 5: Ambient / Autonomous Mode

> The agent keeps working when you step away. Idle detection triggers background iterations on the last topic. Findings auto-inject into your next query.

📖 Full explainer: see the companion page in `content/` of this repo.  
💻 Standalone script: `code/step-04-ambient/agent.py`  
🔗 Workshop: https://github.com/strands-agents/samples/tree/main/python/01-learn/18-self-improving-agents

---


# AIM308 — Module 4: Autonomous (Ambient Mode)

In [ ]:
%pip install -q strands-agents strands-agents-tools bedrock-agentcore

In [ ]:
import os
import boto3
ssm = boto3.client('ssm')
MODEL_ID = ssm.get_parameter(Name='/aim308/model-id')['Parameter']['Value']  # Claude Sonnet 4.5
REGION = ssm.get_parameter(Name='/aim308/region')['Parameter']['Value']
os.environ["AWS_REGION"] = REGION
# Ambient mode runs the agent on a background thread, which cannot answer the
# interactive tool-consent prompt. Bypass it so shell tools don't hang there.
os.environ["BYPASS_TOOL_CONSENT"] = os.getenv("BYPASS_TOOL_CONSENT", "true")

## The ambient loop

In [ ]:
import threading, time

class AmbientMode:
    def __init__(self, agent_factory, idle_seconds=30, max_iterations=3):
        self.agent_factory = agent_factory
        self.idle_seconds = idle_seconds
        self.max_iterations = max_iterations
        self.last_interaction = time.time()
        self.last_query = None
        self.pending_result = None
        self.running = False
        self.iterations = 0

    def record_interaction(self, query):
        self.last_interaction = time.time()
        self.last_query = query
        self.iterations = 0

    def consume_pending(self):
        result, self.pending_result = self.pending_result, None
        return result

    def _loop(self):
        while self.running:
            idle = time.time() - self.last_interaction
            if idle > self.idle_seconds and self.iterations < self.max_iterations and self.last_query:
                prompt = (
                    f"User's last query was: '{self.last_query}'. "
                    f"Continue exploring. Find edge cases, related topics, "
                    f"improvements. Be proactive."
                )
                agent = self.agent_factory()
                result = agent(prompt)
                self.pending_result = str(result)
                self.iterations += 1
            time.sleep(5)

    def start(self):
        self.running = True
        threading.Thread(target=self._loop, daemon=True).start()

## Use it: Ask a question → wait 15s → consume pending results.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands_tools import shell

def factory():
    return Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=REGION),
        tools=[shell]
    )

# Start ambient mode
amb = AmbientMode(factory, idle_seconds=15, max_iterations=2)
amb.start()

# Ask an initial question
query = "What are the top 3 techniques to speed up LLM inference?"
initial_agent = factory()
initial_agent(query)

# Record the interaction so ambient mode can continue exploring
amb.record_interaction(query)

In [ ]:
# wait ~40 seconds, then run this cell

import time

time.sleep(40)
pending = amb.consume_pending()
print(f"Ambient produced {len(pending or '')} chars\n")
print((pending or "")[:800])

Next: **[05_deploy.ipynb](05_deploy.ipynb)**